# Supplementary notebook: R-index calculations for biocircuits

This notebook is the executable supplement for the IFAC 2026 manuscript **"Evaluating valid parameter regimes for biocircuits"**.  It follows the final six-page manuscript (`root_6pages_revised.pdf`) and consolidates the working code from `Hill_2_diff_structure.ipynb` and `NFBLE_competitive.ipynb`.

The goal is to make the paper's regime calculations reproducible in one place:

- define the Realizability Index (R-index) as an asymptotic solid-angle fraction in log-parameter space;
- compute the Michaelis-Menten validity regimes;
- compare sequential and dimer mechanisms for Hill-like input-output maps;
- identify competitive-binding negative-feedback regimes that realize adaptation of `Astar` and/or `tAstar`.

All numerical parameters below are in `log10` coordinates unless stated otherwise.  Monte Carlo volume estimates are stochastic; increase `VOLUME_REL_TOL` or `VOLUME_TIME_LIMIT` when reproducing final manuscript numbers.

In [ ]:
begin
    using Pkg
    Pkg.activate(@__DIR__)
end

using LinearAlgebra
using SparseArrays
using Statistics
using CairoMakie
using LaTeXStrings
using Polyhedra
using BindingAndCatalysis

const VOLUME_REL_TOL = 0.02
const VOLUME_TIME_LIMIT = 60.0

volume_mean(v) = v.mean
volume_stderr(v) = sqrt(v.var)
volume_sum(vs) = sum(volume_mean, vs)

function volume_summary(v; digits=4)
    (; mean=round(volume_mean(v), digits=digits), stderr=round(volume_stderr(v), digits=digits))
end

function volume_summary(vs::AbstractVector; digits=4)
    (; mean=round(volume_sum(vs), digits=digits), stderr=round(sqrt(sum(v.var for v in vs)), digits=digits))
end

function eliminate_parameter(poly, parameter_index::Integer)
    projected = eliminate(poly, BitSet(parameter_index))
    detecthlinearity!(projected)
    removevredundancy!(projected)
    return projected
end

function sym_index(symbols, name::Symbol)
    idx = findfirst(s -> string(s) == String(name), symbols)
    isnothing(idx) && error("symbol $name not found in $(symbols)")
    return idx
end

function logparam(logqK, model, name::Symbol)
    return logqK[sym_index(qK_symbol(model), name)]
end

## 1. R-index and the elementary binding network

For a regime polyhedron described by linear inequalities in log parameters,

```math
\sum_i \alpha_{ij} z_i + \beta_j \ge 0,
```

the R-index uses the recession cone obtained by dropping the offsets `beta_j`.  Its value is the solid-angle fraction of that cone.  In the code this is estimated by Gaussian sampling, matching the manuscript definition.

The elementary binding equilibrium

```math
E + S \rightleftharpoons C,\qquad ES = KC,
```

with conserved totals `tE = E + C` and `tS = S + C` has four species-space dominance choices.  In the natural parameter space `(tE, tS, K)`, the three full-dimensional regimes each occupy one third of the asymptotic volume; the equality-like regime `(C, C)` has zero R-index.

In [ ]:
function elementary_binding_model()
    model = Bnc(
        N = [1 1 -1],
        x_sym = [:E, :S, :C],
        q_sym = [:tE, :tS],
        K_sym = [:K],
    )
    find_all_regimes!(model)
    return model
end

mm_model = elementary_binding_model()
mm_regimes = get_regimes(mm_model)
mm_regime_volumes = get_volumes(mm_model; rel_tol=VOLUME_REL_TOL, time_limit=VOLUME_TIME_LIMIT, show_progress=false)

mm_regime_table = [
    (; regime=i, perm=get_perm(rgm), volume=volume_summary(v))
    for (i, (rgm, v)) in enumerate(zip(mm_regimes, mm_regime_volumes))
]
mm_regime_table

## 2. Michaelis-Menten validity regimes

Under total quasi-steady state, the Michaelis-Menten complex concentration is

```math
C_{MM} = \frac{t_E t_S}{K_m + t_S}.
```

The condition usually used in textbooks, `tS >> tE`, is sufficient but not necessary.  Regime analysis of the same binding equations shows that the Michaelis-Menten expression is valid in regimes `(E, S)` and `(C, S)`.  Treating `(tE, tS, Km)` as free parameters gives R-index `2/3`.  If `tS` is the operating input and the design parameters are only `(tE, Km)`, the validity condition reduces to `Km >> tE`, giving R-index `1/2`.

In [ ]:
mm_valid_regimes = [
    get_regime(mm_model, [1, 2]),  # tE approx E, tS approx S
    get_regime(mm_model, [3, 2]),  # tE approx C, tS approx S
]

mm_valid_polys = get_polyhedron.(mm_valid_regimes)
mm_rindex_all_parts = calc_volume(
    mm_valid_polys;
    rel_tol=VOLUME_REL_TOL,
    time_limit=VOLUME_TIME_LIMIT,
    asymptotic=true,
)

mm_free_tS_poly = eliminate_parameter(
    intersect(mm_valid_polys...),
    locate_sym_qK(mm_model, :tS),
)
mm_rindex_free_tS = calc_volume(
    mm_free_tS_poly;
    rel_tol=VOLUME_REL_TOL,
    time_limit=VOLUME_TIME_LIMIT,
    asymptotic=true,
)

(; R_all_parameters=volume_summary(mm_rindex_all_parts), R_free_tS=volume_summary(mm_rindex_free_tS))

The following cell reproduces the diagnostic comparison used in the manuscript.  The exact complex concentration is obtained analytically from

```math
(t_E-C)(t_S-C)=K_m C.
```

The heatmap shows where the Michaelis-Menten formula over- or under-estimates the exact complex.

In [ ]:
function exact_complex(tE, tS, K)
    b = tE + tS + K
    return (b - sqrt(b^2 - 4tE*tS)) / 2
end

michaelis_menten_complex(tE, tS, K) = tE * tS / (K + tS)

function plot_mm_error(; logK=0.0, range_log=(-6.0, 6.0), n=250)
    logS_vals = collect(range(range_log..., length=n))
    logE_vals = collect(range(range_log..., length=n))
    K = exp10(logK)

    diff = [
        log10(michaelis_menten_complex(exp10(logE), exp10(logS), K)) -
        log10(exact_complex(exp10(logE), exp10(logS), K))
        for logS in logS_vals, logE in logE_vals
    ]

    cmap = vcat(
        cgrad([:blue, :black], 100; rev=true).colors.colors,
        cgrad([:blue, "#D9D9D9", :red], 200).colors.colors,
        cgrad([:red, :black], 100).colors.colors,
    )

    fig = Figure(size=(620, 520), backgroundcolor=:white)
    ax = Axis(
        fig[1, 1];
        xlabel=L"\log_{10} t_S",
        ylabel=L"\log_{10} t_E",
        title=L"\log_{10}(C_{MM}/C_{exact})",
    )
    hm = heatmap!(ax, logS_vals, logE_vals, clamp.(diff, -2, 2); colormap=cmap, colorrange=(-2, 2))
    vlines!(ax, [logK]; color=:gray35, linestyle=:dash)
    hlines!(ax, [logK]; color=:gray35, linestyle=:dash)
    lines!(ax, [logK, last(logS_vals)], [logK, last(logE_vals)]; color=:gray35, linestyle=:dash)
    Colorbar(fig[1, 2], hm)
    return fig
end

plot_mm_error()

## 3. Hill functions from different binding structures

For two binding sites, the final manuscript compares two mechanisms:

- **sequential binding:** `E + S <-> C1`, `C1 + S <-> C2`;
- **dimer binding:** `S + S <-> CS`, `E + CS <-> C2`.

Both can locally produce a Hill-like expression

```math
C_2 \approx \frac{t_E t_S^2}{K_1K_2+t_S^2}
```

or the analogous dimer expression with `KS*KE`.  The R-index depends on both the binding structure and on whether `tS` is treated as a tunable parameter or as a free input that must sweep its full range.

In [ ]:
function sequential_hill_N(n::Integer)
    N = zeros(Int, n, n + 2)
    N[:, 1] .= 1  # S participates in every step
    for i in 1:n
        N[i, i + 1] = 1
        N[i, i + 2] = -1
    end
    return N
end

function sequential_hill_model(n::Integer)
    Bnc(
        N = sequential_hill_N(n),
        x_sym = vcat([:S, :E], Symbol.("C" .* string.(1:n))),
        q_sym = [:tS, :tE],
        K_sym = Symbol.("K" .* string.(1:n)),
    )
end

function dimer_hill_model()
    Bnc(
        N = [
            0 2 -1  0
            1 0  1 -1
        ],
        x_sym = [:E, :S, :CS, :C2],
        q_sym = [:tE, :tS],
        K_sym = [:KS, :KE],
    )
end

function rindex_sequential_hill(n::Integer; rel_tol=VOLUME_REL_TOL, time_limit=VOLUME_TIME_LIMIT)
    model = sequential_hill_model(n)
    find_all_regimes!(model)

    source = get_regime(model, [1, 2])      # tS approx S, tE approx E
    sink = get_regime(model, [1, n + 2])    # tS approx S, tE approx Cn
    polys = get_polyhedron.([source, sink])

    r_all_parts = calc_volume(polys; rel_tol, time_limit, asymptotic=true)
    free_tS_poly = eliminate_parameter(intersect(polys...), locate_sym_qK(model, :tS))
    r_free_tS = calc_volume(free_tS_poly; rel_tol, time_limit, asymptotic=true)

    return (; model, source, sink, R_all_parts=r_all_parts, R_free_tS=r_free_tS)
end

function rindex_dimer_hill2(; rel_tol=VOLUME_REL_TOL, time_limit=VOLUME_TIME_LIMIT)
    model = dimer_hill_model()
    find_all_regimes!(model)

    valid = [
        get_regime(model, [1, 2]),  # tE approx E,  tS approx S
        get_regime(model, [4, 2]),  # tE approx C2, tS approx S
        get_regime(model, [4, 3]),  # tE approx C2, tS approx CS
    ]
    polys = get_polyhedron.(valid)
    r_all_parts = calc_volume(polys; rel_tol, time_limit, asymptotic=true)

    # The free-tS comparison uses the source and the saturated free-S branch;
    # the CS-dominated branch shares the same saturated C2 expression.
    free_tS_poly = eliminate_parameter(
        intersect(get_polyhedron.([valid[1], valid[2]])...),
        locate_sym_qK(model, :tS),
    )
    r_free_tS = calc_volume(free_tS_poly; rel_tol, time_limit, asymptotic=true)

    return (; model, valid, R_all_parts=r_all_parts, R_free_tS=r_free_tS)
end

seq2_result = rindex_sequential_hill(2)
dimer2_result = rindex_dimer_hill2()

hill2_summary = [
    (; model="sequential n=2", R_all=volume_summary(seq2_result.R_all_parts), R_free_tS=volume_summary(seq2_result.R_free_tS)),
    (; model="dimer n=2", R_all=volume_summary(dimer2_result.R_all_parts), R_free_tS=volume_summary(dimer2_result.R_free_tS)),
]
hill2_summary

For the sequential `n`-site network, the manuscript uses precomputed R-index values for `n = 1:13`.  The cell below records those values and plots Fig. 7-style bars.  The function `rindex_sequential_hill(n)` above can be used to recompute any row at the desired Monte Carlo tolerance.

In [ ]:
sequential_hill_rindex_free_tS = [
    0.5005694866185488,
    0.25002611726417034,
    0.16663234614601632,
    0.1249886037394363,
    0.09997770337194163,
    0.08327254375511496,
    0.07147923145738,
    0.06251352827741102,
    0.05562429452692954,
    0.05001751579502254,
    0.04544599819296912,
    0.041677861394426786,
    0.03845511815904901,
]

sequential_hill_source_sink_parts = [
    (0.33339177143099336, 0.3333799532550712),
    (0.26051117454422373, 0.3020545798201327),
    (0.23526864135636902, 0.2892914536240215),
    (0.2225540789479319, 0.282096207864127),
    (0.21525667332125986, 0.27785958533432553),
    (0.21012697334726177, 0.2745494966724338),
    (0.20672656260796168, 0.2723237742338685),
    (0.20408580911614524, 0.27081046927136),
    (0.20212040422975006, 0.2696058938142354),
    (0.2004046201872282, 0.26832789428179565),
    (0.19912601005129307, 0.2676426313765343),
    (0.19815142048326206, 0.2669145184328566),
    (0.19710491487636922, 0.26619187718530907),
]

sequential_hill_rindex_all = [sum(v) for v in sequential_hill_source_sink_parts]

fig = Figure(size=(820, 620), backgroundcolor=:white)
ns = 1:13
ax1 = Axis(fig[1, 1]; xlabel="n", ylabel="R-index", title="Sequential Hill: all parameters", xticks=ns)
barplot!(ax1, ns, sequential_hill_rindex_all; color="#E2C867")
text!(ax1, ns, sequential_hill_rindex_all .+ 0.01; text=string.(round.(sequential_hill_rindex_all, digits=3)), align=(:center, :bottom), fontsize=12)
ylims!(ax1, 0, 0.75)

ax2 = Axis(fig[2, 1]; xlabel="n", ylabel="R-index", title="Sequential Hill: tS treated as free input", xticks=ns)
barplot!(ax2, ns, sequential_hill_rindex_free_tS; color="#5B8FF9")
text!(ax2, ns, sequential_hill_rindex_free_tS .+ 0.01; text=string.(round.(sequential_hill_rindex_free_tS, digits=3)), align=(:center, :bottom), fontsize=12)
ylims!(ax2, 0, 0.60)
fig

The optional cell below reproduces the two-dimensional Hill-error slices from the development notebook.  It is off by default because each slice solves many binding equilibria.  Set `RUN_HILL_SLICES = true` to generate the heatmaps.

In [ ]:
RUN_HILL_SLICES = false

function hill2_sequential_error(model, logx, logqK)
    tS = exp10(logparam(logqK, model, :tS))
    tE = exp10(logparam(logqK, model, :tE))
    K1 = exp10(logparam(logqK, model, :K1))
    K2 = exp10(logparam(logqK, model, :K2))
    log_C_hill = log10(tE * tS^2 / (K1 * K2 + tS^2))
    return clamp(log_C_hill - logx[locate_sym_x(model, :C2)], -2, 2)
end

function hill2_dimer_error(model, logx, logqK)
    tS = exp10(logparam(logqK, model, :tS))
    tE = exp10(logparam(logqK, model, :tE))
    KS = exp10(logparam(logqK, model, :KS))
    KE = exp10(logparam(logqK, model, :KE))
    log_C_hill = log10(tE * tS^2 / (KS * KE + tS^2))
    return clamp(log_C_hill - logx[locate_sym_x(model, :C2)], -2, 2)
end

function hill_error_colormap()
    vcat(
        cgrad([:blue, :black], 100; rev=true).colors.colors,
        cgrad([:blue, "#D9D9D9", :red], 200).colors.colors,
        cgrad([:red, :black], 100).colors.colors,
    )
end

if RUN_HILL_SLICES
    seq2_model = sequential_hill_model(2)
    find_all_regimes!(seq2_model)
    fig_seq, ax_seq, _ = plot_binding_regime_partition(
        seq2_model;
        axes=[:tS, :tE],
        fixed=Dict(:K1 => 3.0, :K2 => -3.0),
        ranges=(-6.0, 6.0),
        n=220,
        value_func=(logx, logqK) -> hill2_sequential_error(seq2_model, logx, logqK),
        colormap=hill_error_colormap(),
        colorrange=(-2, 2),
        method=:homotopy,
    )
    display(fig_seq)

    dimer_model = dimer_hill_model()
    find_all_regimes!(dimer_model)
    fig_dimer, ax_dimer, _ = plot_binding_regime_partition(
        dimer_model;
        axes=[:tS, :tE],
        fixed=Dict(:KS => 3.0, :KE => -3.0),
        ranges=(-6.0, 6.0),
        n=220,
        value_func=(logx, logqK) -> hill2_dimer_error(dimer_model, logx, logqK),
        colormap=hill_error_colormap(),
        colorrange=(-2, 2),
        method=:homotopy,
    )
    display(fig_dimer)
end

## 4. Competitive negative-feedback adaptation

The competitive enzymatic negative-feedback loop uses the binding network

```math
I + A \rightleftharpoons C_1,
\quad A^* + B^* \rightleftharpoons C_2,
\quad A^* + B \rightleftharpoons C_3,
\quad E + B^* \rightleftharpoons C_4.
```

The catalytic fluxes are

```math
\dot t_{A^*} = k_{A1}C_1-k_{A2}C_2,
\qquad
\dot t_{B^*} = k_{B1}C_3-k_{B2}C_4.
```

A regime is counted as adaptive when it is locally stable, full-dimensional in the combined `wKk` parameter space, input-responsive, and has steady-state invariance with respect to the input `tI`.

In [ ]:
function competitive_nfble_model()
    model = let
        x_sym = [:I, :A, :Astar, :B, :Bstar, :E, :C1, :C2, :C3, :C4]
        q_sym = [:tI, :tA, :tAstar, :tB, :tBstar, :tE]
        K_sym = [:KA1, :KA2, :KB1, :KB2]

        N = [
            1 1 0 0 0 0 -1  0  0  0
            0 0 1 0 1 0  0 -1  0  0
            0 0 1 1 0 0  0  0 -1  0
            0 0 0 0 1 1  0  0  0 -1
        ]

        Bnc(N=N, x_sym=x_sym, q_sym=q_sym, K_sym=K_sym)
    end

    Pi_cat = diagm(ones(Int, 4))
    Gamma_cat = [
         1 -1  0  0
        -1  1  0  0
         0  0  1 -1
         0  0 -1  1
    ]

    update_catalysis!(
        model;
        Π=Pi_cat,
        Γ=Gamma_cat,
        x_picked=[:C1, :C2, :C3, :C4],
        q_picked=[:tAstar, :tA, :tBstar, :tB],
        w_sym=[:tAtotal, :tBtotal],
        k_sym=[:kA1, :kA2, :kB1, :kB2],
    )

    return model
end

function ss_invariant_flags(rgm)
    model = get_binding_network(rgm)
    input_idx = locate_sym_wKk(model, :tI)

    H = get_H(rgm)
    x_Astar = locate_sym_x(model, :Astar)
    invariant_Astar = abs(H[x_Astar, input_idx]) < 1e-6

    F, _ = get_qcat_F_F0(rgm)
    q_tAstar = locate_sym_qcat(model, :tAstar)
    invariant_tAstar = abs(F[q_tAstar, input_idx]) < 1e-6

    return invariant_Astar, invariant_tAstar
end

function input_response_flags(rgm; threshold=0.5)
    model = get_binding_network(rgm)
    binding_rgm = get_binding_regime(rgm)

    H_binding = is_singular(binding_rgm) ? get_H_numerically(binding_rgm) : get_H(binding_rgm)

    x_C1 = locate_sym_x(model, :C1)
    q_tI = locate_sym_qK(model, :tI)
    q_tAstar = locate_sym_qK(model, :tAstar)
    x_Astar = locate_sym_x(model, :Astar)

    response_tAstar = H_binding[x_C1, q_tI] >= threshold
    response_Astar = response_tAstar && abs(H_binding[x_Astar, q_tAstar]) >= threshold

    return response_Astar, response_tAstar
end

nfble = competitive_nfble_model()
match_regimes!(nfble)

nfble_regimes = get_bnc_regimes(nfble; singular=false)
stable_nfble_regimes = filter(is_stable, nfble_regimes)

good_Astar_regimes = filter(stable_nfble_regimes) do rgm
    ss_invariant_flags(rgm)[1] && input_response_flags(rgm)[1]
end

good_tAstar_regimes = filter(stable_nfble_regimes) do rgm
    ss_invariant_flags(rgm)[2] && input_response_flags(rgm)[2]
end

valid_nfble_regimes = union(good_Astar_regimes, good_tAstar_regimes)
shared_nfble_regimes = intersect(good_Astar_regimes, good_tAstar_regimes)

valid_nfble_volumes = get_volumes(
    valid_nfble_regimes;
    rel_tol=VOLUME_REL_TOL,
    time_limit=VOLUME_TIME_LIMIT,
    show_progress=false,
)

function sum_selected_volumes(regimes, selected, volumes)
    idxs = findall(rgm -> rgm in selected, regimes)
    return volume_summary(volumes[idxs])
end

nfble_summary = (;
    total_mixed_regimes=n_bnc_regimes(nfble),
    full_dimensional_regimes=length(nfble_regimes),
    stable_full_dimensional_regimes=length(stable_nfble_regimes),
    Astar_valid_regimes=length(good_Astar_regimes),
    tAstar_valid_regimes=length(good_tAstar_regimes),
    shared_valid_regimes=length(shared_nfble_regimes),
    R_Astar=sum_selected_volumes(valid_nfble_regimes, good_Astar_regimes, valid_nfble_volumes),
    R_tAstar=sum_selected_volumes(valid_nfble_regimes, good_tAstar_regimes, valid_nfble_volumes),
)

nfble_summary

The final manuscript reports, using tighter volume estimates,

```text
Astar adaptation:   14 regimes, R = 0.0194 +/- 0.00004
tAstar adaptation:  20 regimes, R = 0.03291 +/- 0.00005
shared regimes:      8 regimes
```

The next cell extracts the largest shared adaptive regime and prints the reduced expressions.  This is the regime-level mechanism highlighted in the paper: `tBstar` acts as an integral-like state and the steady-state values of both `Astar` and `tAstar` are independent of `tI`.

In [ ]:
shared_volume_means = [
    volume_mean(valid_nfble_volumes[findfirst(==(rgm), valid_nfble_regimes)])
    for rgm in shared_nfble_regimes
]

largest_shared_nfble_regime = shared_nfble_regimes[argmax(shared_volume_means)]

println("largest shared regime id: ", get_idx(largest_shared_nfble_regime))
println("estimated R-index of this regime: ", round(maximum(shared_volume_means), digits=5))

display(show_expression_qcat(largest_shared_nfble_regime))
display(show_expression_x(largest_shared_nfble_regime))
display(show_condition_wKk(largest_shared_nfble_regime))

### Dynamic verification

The following simulation uses the parameter point reported in the final manuscript Fig. 10 caption.  The vector order is `wKk_sym(nfble)`:

```text
[tAtotal, tBtotal, tI, tE, KA1, KA2, KB1, KB2, kA1, kA2, kB1, kB2]
```

The input `tI` starts at `10^1.1`, steps to `10^2.1` at `t = 8000`, and steps to `10^0.1` at `t = 25000`.

In [ ]:
p_adaptation = [
    2.9,   # tAtotal
    5.8,   # tBtotal
    1.1,   # initial tI
   -1.3,   # tE
   -1.1,   # KA1
    2.9,   # KA2
    2.9,   # KB1
    0.0,   # KB2
   -3.4,   # kA1
    0.0,   # kA2
   -2.2,   # kB1
    0.0,   # kB2
]

assigned_regime_id = assign_bnc_regime_wKk(nfble, p_adaptation)
adaptation_regime = get_bnc_regime(nfble, assigned_regime_id)

F_adapt, F0_adapt = get_qcat_F_F0(adaptation_regime)
logqcat0 = Vector{Float64}(F_adapt * p_adaptation .+ F0_adapt)

logtI_step(t) = t < 8_000 ? 1.1 : t < 25_000 ? 2.1 : 0.1
savegrid = range(0.0, 80_000.0, length=320)

adapt_Astar = simulate_adaptation(
    nfble;
    p=p_adaptation,
    logtI=logtI_step,
    logqcat0=logqcat0,
    tspan=(0.0, 80_000.0),
    observe=:Astar,
    saveat=savegrid,
    tstops=[8_000.0, 25_000.0],
    maxiters=300_000,
)

adapt_tAstar = merge(adapt_Astar, (;
    logobserve=vec(adapt_Astar.logqcat[locate_sym_qcat(nfble, :tAstar), :]),
    observe=:tAstar,
))

fig = Figure(size=(760, 620), backgroundcolor=:white)
ax0 = Axis(fig[1, 1]; ylabel=L"\log_{10} t_I")
lines!(ax0, adapt_Astar.t, adapt_Astar.logtI; color=:black)
hidexdecorations!(ax0, grid=false)

ax1 = Axis(fig[2, 1]; ylabel=L"\log_{10} t_{A^*}")
lines!(ax1, adapt_tAstar.t, adapt_tAstar.logobserve; color="#0072B2")
hlines!(ax1, [logqcat0[locate_sym_qcat(nfble, :tAstar)]]; color=:red, linestyle=:dash)
hidexdecorations!(ax1, grid=false)

ax2 = Axis(fig[3, 1]; xlabel="time", ylabel=L"\log_{10} A^*")
lines!(ax2, adapt_Astar.t, adapt_Astar.logobserve; color="#D55E00")
H_adapt, H0_adapt = get_H_H0(adaptation_regime)
initial_logx = Vector{Float64}(H_adapt * p_adaptation .+ H0_adapt)
hlines!(ax2, [initial_logx[locate_sym_x(nfble, :Astar)]]; color=:red, linestyle=:dash)

fig

### Optional constrained R-index calculation

The paper also discusses why the adaptive regimes are hard to find by brute-force parameter sweeps in a finite experimental box.  The unconstrained R-index above samples directions in log-parameter space.  A constrained estimate instead samples a finite box.  This can be very slow for the competitive feedback model because the successful fraction is around `10^-7` in the box used for the comparison, so the code is disabled by default.

In [ ]:
RUN_CONSTRAINED_NFBLE_VOLUME = false

if RUN_CONSTRAINED_NFBLE_VOLUME
    protein_range = (2.0, 4.0)   # protein totals, log10 scale
    constant_range = (-3.0, 4.0) # binding and catalytic constants, log10 scale

    lower = [fill(protein_range[1], 4); fill(constant_range[1], 8)]
    upper = [fill(protein_range[2], 4); fill(constant_range[2], 8)]

    constrained_Astar_parts = calc_volume(
        get_polyhedron.(good_Astar_regimes);
        sampler=:uniform_box,
        log_lower=lower,
        log_upper=upper,
        rel_tol=0.2,
        time_limit=600.0,
        asymptotic=false,
    )

    volume_summary(constrained_Astar_parts)
end

## 5. Manuscript summary table

The table below collects the R-index values used in the final manuscript.  Values labeled "computed here" are produced by the code above at the current notebook tolerance; values labeled "manuscript" are the tighter estimates reported in the paper.

In [ ]:
si_summary_table = [
    (; case="Michaelis-Menten, all parameters", R="2/3", source="analytic / computed here"),
    (; case="Michaelis-Menten, tS free", R="1/2", source="analytic / computed here"),
    (; case="Sequential Hill n=2, all parameters", R=string(round(sequential_hill_rindex_all[2], digits=3)), source="manuscript precompute"),
    (; case="Sequential Hill n=2, tS free", R=string(round(sequential_hill_rindex_free_tS[2], digits=3)), source="manuscript precompute"),
    (; case="Dimer Hill n=2, all parameters", R="0.646", source="manuscript / computed here"),
    (; case="Dimer Hill n=2, tS free", R="0.250", source="manuscript / computed here"),
    (; case="Competitive NFBL, Astar adaptation", R="0.0194 +/- 0.00004", source="manuscript"),
    (; case="Competitive NFBL, tAstar adaptation", R="0.03291 +/- 0.00005", source="manuscript"),
]

si_summary_table